# grad-accumulate-on-leaf composite — cx23: reverse-pass driver with leaf-grad accumulation (rebind, not +=)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `grad-accumulate-on-leaf`, `backprop-pop-outgrad-loop`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "grad-accumulate-on-leaf"
DD_ATOM_IDS = ["grad-accumulate-on-leaf", "backprop-pop-outgrad-loop"]
DD_SUBTOPICS = ["Backprop: Grad accumulate on leaf", "Backprop: backprop pop-outgrad loop"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing leaf-grad accumulation with the reverse-pass driver

When the reverse pass reaches a node, it splits two ways:

- **non-leaf** (`recipe is not None`) — dispatch to back_fns, route the
  parent grads through the scratch `grads` dict.
- **leaf** (`recipe is None`) — write the accumulated grad to   `node.grad` for the optimizer to consume.

The LEAF branch is where `grad-accumulate-on-leaf` lives:

```python
for node in sorted_graph:                        # pop-outgrad loop
    if id(node) not in grads: continue
    grad_out = grads.pop(id(node))
    if node.recipe is None:                      # LEAF — accumulate
        if node.grad is None:
            node.grad = grad_out                 # first touch
        else:
            node.grad = node.grad + grad_out     # rebind, NOT +=
        continue
    # ... non-leaf dispatch ...
```

**Critical: rebind, not in-place.** `node.grad = node.grad + g` creates a new tensor and rebinds. `node.grad += g` mutates the
EXISTING grad tensor — and if `optimizer.step()` is holding a
reference to that tensor, the snapshot silently changes under it.
Rebinding leaves external references alone.

**Why accumulate, not overwrite.** A single leaf can be a parent of
multiple consumers (e.g. `y = w * w` makes `w` a parent of `y` TWICE
via argnums 0 AND 1; weight-tied embedding/unembedding makes a tensor
a parent of two separate downstream ops). Each path's contribution is
summed into the leaf's grad.

### Composite Exercise — reverse-pass driver with leaf-grad accumulation (rebind, not +=)

**Atoms exercised together**: `grad-accumulate-on-leaf`, `backprop-pop-outgrad-loop`

Implement `cx23_backprop(end_node, end_grad, sorted_graph, back_funcs)` — the reverse-pass driver, focused on the LEAF accumulation step. Two atoms compose:

**OUTER** (`backprop-pop-outgrad-loop`) — pop `grad_out` from the scratch `grads` dict for each node in `sorted_graph`.

**LEAF branch** (`grad-accumulate-on-leaf`) — when `node.recipe is None`, write to `node.grad`:
- If `node.grad is None`: set `node.grad = grad_out` (first touch).
- Else: `node.grad = node.grad + grad_out` — **REBIND** with `+`, NOT in-place `+=`.

For non-leaf nodes, dispatch each parent via `back_funcs[(recipe.func, argnum)]` and accumulate into `grads`.

**The rebind-vs-in-place test is load-bearing.** The test holds an external reference to a leaf's grad BEFORE a second backward call. After the second call, the OLD reference MUST be unchanged (rebind semantics) — the leaf MUST hold a NEW tensor object.

**The `y = w * w` accumulation test** confirms the multi-path case: `w` is a parent at argnum 0 AND argnum 1, so two contributions sum into `w.grad` → final value = `2w`.

Return `None`.

In [ ]:
def cx23_backprop(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            # LEAF — accumulate; REBIND with `+`, not in-place `+=`.
            if node.grad is None:
                node.grad = grad_out                  # first touch
            else:
                node.grad = node.grad + grad_out      # rebind!
            continue
        # non-leaf: dispatch every parent
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            gp = back_fn(grad_out, node.array,
                         *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
    return None


<details><summary>Show solution — cx23</summary>

```python
def cx23_backprop(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            # LEAF — accumulate; REBIND with `+`, not in-place `+=`.
            if node.grad is None:
                node.grad = grad_out                  # first touch
            else:
                node.grad = node.grad + grad_out      # rebind!
            continue
        # non-leaf: dispatch every parent
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            gp = back_fn(grad_out, node.array,
                         *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
    return None
```

**`+` rebinds, `+=` mutates.** The optimizer's `p.data -= lr * p.grad` line typically reads `p.grad` mid-step. If our backward MUTATES the grad tensor in place via `+=`, that live reference silently changes value. Rebinding produces a fresh tensor; the optimizer's snapshot is safe.

**First-touch sets directly to skip an allocation.** `node.grad = grad_out` (no `+ 0`) avoids an unnecessary `t.zeros_like(grad_out)` and an add on the first call per leaf. Across a model with K parameters, that's K saved allocations per step.

**Accumulation is why `zero_grad()` exists.** Because `cx23_backprop` ALWAYS adds (never overwrites), the previous step's grads stay around forever unless explicitly cleared. PyTorch's `optimizer.zero_grad()` exists precisely to clear `.grad` between training steps.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx23'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx23',
        'subtopics': ["Backprop: Grad accumulate on leaf", "Backprop: backprop pop-outgrad loop"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()